In [ ]:
import pandas as pd
import copy
from experiments_2024 import DATASETS_PATH

In [ ]:
KWH_PER_THOUSAND_BTU = 0.293071

# Nonresponse Adjusted Weight
weight = "FINALWT"

# PBA
building_cat = "PBA"
CBECS_BUILDING_CATEGORIES = {
    1: "Vacant",
    2: "Office",
    4: "Laboratory",
    5: "Nonrefrigerated warehouse",
    6: "Food sales",
    7: "Public order and safety",
    8: "Outpatient health care",
    11: "Refrigerated warehouse",
    12: "Religious worship",
    13: "Public assembly",
    14: "Education",
    15: "Food service",
    16: "Inpatient health care",
    17: "Nursing",
    18: "Lodging",
    23: "Strip shopping center",
    24: "Enclosed mall",
    25: "Retail other than mall",
    26: "Service",
    91: "Other",
}

# Square footage
square_feet = "SQFT"

# Variable air volume (VAV) system; 1=Yes 2=No, Missing=Not applicable
vav = "VAV"

# Does this building have a building automation system,
# which may also be referred to as BAS, smart building controls,
# an energy management and control system, or EMCS?
# 1=Yes 2=No, Missing=Not applicable
bas = "EMCS"

# Derived variable: Annual major fuels consumption (thous Btu)
total_energy = "MFBTU"

# Building/energy supplier variable: Annual electricity consumption (thous Btu)
total_electricity = "ELBTU"

# Modeled variable: Major fuels heating use (thous Btu)
heating_energy = "MFHTBTU"

# Modeled variable: Electricity heating use (thous Btu)
heating_electricity = "ELHTBTU"

# Modeled variable: District heat heating use (thous Btu)
heating_district = "DHHTBTU"

# Modeled variable: Major fuels cooling use (thous Btu)
cooling_energy = "MFCLBTU"

# Modeled variable: Electricity cooling use (thous Btu)
cooling_electricity = "ELCLBTU"

# Modeled variable: District heat cooling use (thous Btu)
cooling_district = "DHCLBTU"

# Modeled variable: Major fuels ventilation use (thous Btu)
ventilation_energy = "MFVNBTU"

# Modeled variable: Electricity ventilation use (thous Btu)
ventilation_electricity = "ELVNBTU"

In [ ]:
CBECS = pd.read_csv(DATASETS_PATH / "cbecs2018_final_public.csv")

# Summary

In [ ]:
summary = pd.DataFrame(
    index=[
        "Number of Buildings (Million)",
        "Square Feet (Billion)",
        "Energy Consumption (TWh)",
        "Energy Consumption - Cooling (TWh)",
        "Energy Consumption - Heating (TWh)",
        "Energy Consumption - Ventilation (TWh)",
        "Energy Consumption - HVAC (TWh)",
        "Electricity Consumption (TWh)",
        "Electricity Consumption - Cooling (TWh)",
        "Electricity Consumption - Heating (TWh)",
        "Electricity Consumption - Ventilation (TWh)",
        "Electricity Consumption - HVAC (TWh)",
    ],
    columns=[
        "All Commercial Buildings",
        "Commercial Buildings With BAS",
        "Commercial Buildings With AH-VAVs",
        "Commercial Buildings With BAS & AH-VAVs",
    ],
)


masks = {
    "All Commercial Buildings": slice(None),
    "Commercial Buildings With BAS": CBECS[bas] == 1,
    "Commercial Buildings With AH-VAVs": CBECS[vav] == 1,
    "Commercial Buildings With BAS & AH-VAVs": (CBECS[bas] == 1) & (CBECS[vav] == 1),
}

for col in masks:
    summary.loc["Number of Buildings (Million)", col] = (
        CBECS.loc[masks[col], weight].sum() / 1e6
    )

calc_map = {
    "Square Feet (Billion)": square_feet,
    "Energy Consumption (TWh)": total_energy,
    "Energy Consumption - Cooling (TWh)": cooling_energy,
    "Energy Consumption - Heating (TWh)": heating_energy,
    "Energy Consumption - Ventilation (TWh)": ventilation_energy,
    "Electricity Consumption (TWh)": total_electricity,
    "Electricity Consumption - Cooling (TWh)": cooling_electricity,
    "Electricity Consumption - Heating (TWh)": heating_electricity,
    "Electricity Consumption - Ventilation (TWh)": ventilation_electricity,
}


for col in masks:
    for idx in calc_map:
        if idx == "Square Feet (Billion)":
            constant = 1 / 1e9
        else:
            constant = KWH_PER_THOUSAND_BTU / 1e9
        summary.loc[idx, col] = (
            CBECS.loc[masks[col], weight] * CBECS.loc[masks[col], calc_map[idx]]
        ).sum() * constant

summary.loc["Energy Consumption - HVAC (TWh)", :] = (
    summary.loc["Energy Consumption - Cooling (TWh)", :]
    + summary.loc["Energy Consumption - Heating (TWh)", :]
    + summary.loc["Energy Consumption - Ventilation (TWh)", :]
)

summary.loc["Electricity Consumption - HVAC (TWh)", :] = (
    summary.loc["Electricity Consumption - Cooling (TWh)", :]
    + summary.loc["Electricity Consumption - Heating (TWh)", :]
    + summary.loc["Electricity Consumption - Ventilation (TWh)", :]
)

In [ ]:
summary.loc[
    [
        "Number of Buildings (Million)",
        "Square Feet (Billion)",
        "Electricity Consumption (TWh)",
        "Electricity Consumption - HVAC (TWh)",
    ],
    :,
]

In [ ]:
summary